## Text Summarizatoin using the BERT Transformers:


## What is BART ?
1. Bideirectional and Auto Regressie Transformers(BERT + GPT)
2.

In [16]:
!pip install datasets

In [17]:
from datasets import load_dataset

ds = load_dataset("knkarthick/dialogsum")

In [18]:
ds

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 12460
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 500
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 1500
    })
})

In [19]:
ds['train'][1]['dialogue'] # The keywords will be extracted from the dialogue and preprocess them in to produce the summary output

"#Person1#: Hello Mrs. Parker, how have you been?\n#Person2#: Hello Dr. Peters. Just fine thank you. Ricky and I are here for his vaccines.\n#Person1#: Very well. Let's see, according to his vaccination record, Ricky has received his Polio, Tetanus and Hepatitis B shots. He is 14 months old, so he is due for Hepatitis A, Chickenpox and Measles shots.\n#Person2#: What about Rubella and Mumps?\n#Person1#: Well, I can only give him these for now, and after a couple of weeks I can administer the rest.\n#Person2#: OK, great. Doctor, I think I also may need a Tetanus booster. Last time I got it was maybe fifteen years ago!\n#Person1#: We will check our records and I'll have the nurse administer and the booster as well. Now, please hold Ricky's arm tight, this may sting a little."

In [20]:
ds['train'][1]['summary'] # Summary of the entire dialogue

'Mrs Parker takes Ricky for his vaccines. Dr. Peters checks the record and then gives Ricky a vaccine.'

# Without Fine Tuning:

In [21]:
!pip install transformers

In [22]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Load the tokenizer and model directly
tokenizer = AutoTokenizer.from_pretrained("facebook/bart-large-cnn")
model = AutoModelForSeq2SeqLM.from_pretrained("facebook/bart-large-cnn")

# You can then create a pipeline manually or use the model for inference directly
# For simplicity, let's create a custom summarization function
def generate_summary(text, max_length=150, min_length=40, do_sample=False):
    inputs = tokenizer(text, max_length=1024, truncation=True, return_tensors="pt")
    summary_ids = model.generate(inputs["input_ids"], num_beams=4, max_length=max_length, min_length=min_length, early_stopping=True)
    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

In [23]:
dialogue_to_summarize = ds['train'][1]['dialogue']

print(dialogue_to_summarize)
summary = generate_summary(dialogue_to_summarize)


#Person1#: Hello Mrs. Parker, how have you been?
#Person2#: Hello Dr. Peters. Just fine thank you. Ricky and I are here for his vaccines.
#Person1#: Very well. Let's see, according to his vaccination record, Ricky has received his Polio, Tetanus and Hepatitis B shots. He is 14 months old, so he is due for Hepatitis A, Chickenpox and Measles shots.
#Person2#: What about Rubella and Mumps?
#Person1#: Well, I can only give him these for now, and after a couple of weeks I can administer the rest.
#Person2#: OK, great. Doctor, I think I also may need a Tetanus booster. Last time I got it was maybe fifteen years ago!
#Person1#: We will check our records and I'll have the nurse administer and the booster as well. Now, please hold Ricky's arm tight, this may sting a little.


## With Fine tuning:

In [24]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("facebook/bart-large-cnn")
model = AutoModelForSeq2SeqLM.from_pretrained("facebook/bart-large-cnn")

Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

## Fine-tuning the BART model

To fine-tune the BART model for summarization, we need to:

1.  **Preprocess the dataset**: Convert the dialogue and summary text into token IDs that the model can understand.
2.  **Define training arguments**: Specify parameters like batch size, learning rate, and the number of epochs.
3.  **Create a `Trainer`**: Utilize the Hugging Face `Trainer` API for efficient training and evaluation.

In [25]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("facebook/bart-large-cnn")
model = AutoModelForSeq2SeqLM.from_pretrained("facebook/bart-large-cnn")

Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

In [26]:
# Toenization : Converts the dialogue and summary gets converted into tokens, Assigns each word a token id's
def preprocessing_function(batch):
  source = batch['dialogue']
  target = batch['summary']
  source_ids = tokenizer(source, truncation=True, padding='max_length',max_length = 128)
  target_ids = tokenizer(target, truncation=True, padding='max_length',max_length = 128)

  labels = target_ids['input_ids']
  labels = [[(label if label != tokenizer.pad_token_id else - 100) for label in labels_example] for labels_example in labels]

  return {
      'input_ids': source_ids['input_ids'],
      'attention_mask': source_ids['attention_mask'],
      'labels': labels
  }

In [27]:
df_source = ds.map(preprocessing_function, batched = True)

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

After training, you can save the fine-tuned model and tokenizer.

In [28]:
# training arguments:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir = "/content", # Locatioin to store all the data
    per_device_train_batch_size = 8,
    num_train_epochs = 2,
    remove_unused_columns = True,  # "Id" and "Topics " colummns are not required


)









In [29]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=df_source['train'], # Passing the df_score for training the data:
    eval_dataset=df_source['test']
)

In [30]:
trainer.train()

Step,Training Loss
500,1.591094
1000,1.488247
1500,1.433422


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Step,Training Loss
500,1.591094
1000,1.488247
1500,1.433422
2000,1.083223
2500,1.018975
3000,1.001156


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3116, training_loss=1.2603704528416841, metrics={'train_runtime': 3922.2128, 'train_samples_per_second': 6.354, 'train_steps_per_second': 0.794, 'total_flos': 6750530835578880.0, 'train_loss': 1.2603704528416841, 'epoch': 2.0})

In [31]:
eval_results = trainer.evaluate()

Training Loss,Validation Loss,Step
1.001156,1.662025,3116


In [32]:
eval_results

{'eval_loss': 1.662024736404419}

## Saving the model

In [33]:
model.save_pretrained('/content/model_directory')
tokenizer.save_pretrained('/content/model_directory')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/model_directory/tokenizer_config.json',
 '/content/model_directory/tokenizer.json')

In [36]:
# Input the Custom data :
tokenizer = AutoTokenizer.from_pretrained('/content/model_directory')
model = AutoModelForSeq2SeqLM.from_pretrained('/content/model_directory')

def summary(blog_post):
  inputs = tokenizer(blog_post, max_length=1024, truncation=True, return_tensors='pt')

  # Generate the summary:
  summary_ids = model.generate(inputs['input_ids'], max_length=150, min_length= 40, length_penalty = 2.0, num_beams = 4, early_stopping = True)

  # decode the summary: # Which tokens is assigned to which number
  summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

  return summary

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

In [39]:
blog_post = """
As Yogi Berra famously said, it’s tough to make predictions, especially about the future. But had the baseball legend spent any time observing the UN climate negotiations, he could have safely predicted that climate finance will prove to be a key sticking point at COP29 in Baku at the end of this year.

‘Who will pay and how much?’ are perennial questions at the climate talks, but this year, the discussions about climate finance will be especially prominent. At COP29, Parties to the Paris Agreement must negotiate a new climate finance goal, to replace the existing commitment from 2009 for developed countries to provide US$100 billion climate finance annually from 2020 to 2025 - a commitment that only in 2022 was starting to be fulfilled, according to a recent OECD report.

It is vital that the forthcoming Bonn Climate Change Conference sends the right political signals, and lays the procedural and technical groundwork for an ambitious climate finance deal in Baku.

A pressing need

With global warming already destabilising the climate and devastating people’s lives and livelihoods, the need for finance to reduce greenhouse gas emissions and to adapt to a warming world has never been more pressing.

The sums involved are large. The Paris Agreement’s Global Stocktake process estimates that US$5.8-5.9 trillion is required to implement Nationally Determined Contributions (NDCs) in developing countries up to 2030. They will require US$215-387 billion annually over this period for adaptation. Investments of US$1.5 trillion in renewable energy are required worldwide every year up until 2030, according to IRENA.

But these sums are also affordable and beneficial for developed countries. They should be seen in the context of ongoing investments in energy and other infrastructure: around US$2.3 trillion was invested in energy infrastructure in 2023, of which US$1.74 trillion was in clean energy. These investments will generate strong returns for their investors and reduce the costs for energy consumers.

And, crucially, they should also be seen in the context of the alternative. The latest research estimates that the world economy is already set to face a 19% income reduction within the next 26 years based on the levels of warming we have already locked in. The more we delay and the more the planet heats, the greater the economic costs will be.

Laying the foundations for a new finance goal

While financial resources are beginning to flow, they are not flowing fast enough, and certainly not flowing to those developing countries where need is greatest and access to finance is most challenging.

The UN climate framework provides mechanisms that can enable those flows of climate finance. Back in 2015, parties at the climate talks agreed to establish a “new collective quantified goal” (NCQG) for climate finance. They agreed that the NCQG would be set prior to 2025.

The  ultimate size of the NCQG will be a product of the negotiations, but Parties have agreed it must be a significant increase from the floor of US$100 billion annually. For WWF, it must be needs-based and sufficiently ambitious to meet the scale of the challenge we face, and immediately accessible to help countries that are already facing the chaos of a destabilised climate system.

While developed countries are expected to provide financial and technical support, developing countries also have a role to play. Parties are due to submit revised NDCs in 2025, presenting how they plan to reduce emissions and adapt to climate change. Developing countries have the opportunity to use their NDCs to set out how international climate finance can support them and increase their ambition. To do this, they need to know the finance will be forthcoming.

Adaptation in focus

Historically, climate finance has predominantly been directed towards mitigating climate change through emissions reductions and the conservation of forests as carbon sinks. As the impacts of a changing climate are increasingly felt, finance for adaptation and loss and damage must also be dramatically scaled up. We strongly advocate for at least 50% of public climate finance to be allocated to adaptation, as per Article 9.4 in the Paris Agreement.

Similarly, the Loss and Damage Fund, which was operationalised at COP28, must be sufficiently resourced with new funding to make it fit for purpose. The initial pledges amounted to little more than $700m. To meet expected needs, parties should put arrangements in place to ensure that the fund has a reliable and steady source of funding, of at least US$100 billion per year, as soon as possible. Under NCQG, there should be clear sub goals for mitigation, adaptation and loss and damage.

Fixing financial flows

Public finance will be crucial for the bulk of finance for adaptation, loss and damage and for the transition to a net-zero global economy. Private finance will also have a critical role to play. Currently, more finance is flowing to industries and activities that harm our planet than to the sustainable industries of the future. This has to be reversed. Here, central banks and regulatory agencies have a key role to play in regulating to reduce climate and nature risk within financial systems, and to encourage private finance to underwrite sustainable economic activity.

We know what we need to do, and we have the mechanisms in place, through the Paris Agreement, to make finance flow. What we need is the political will to fund those mechanisms at the scale and speed needed. Bonn needs to send the right political signals that will put governments on course for an ambitious climate finance outcome in Baku. It needs to lay the groundwork to ensure that this “Finance COP” doesn’t short-change the world.

"""

summary = summary(blog_post)
print(f"Summary: {summary}")

Summary: #ClimateFinance will be a key sticking point at the COP29 in Baku. #WWF argues that the need for finance to reduce greenhouse gas emissions and to adapt to a warming world has never been more pressing. They also argue that developing countries also need to play a role to meet the challenge.
